# Micrograph generator

This notebook simulates a full cryo-EM micrograph (with crowded, tiled ice) from a PDB/mmCIF structure. All of the parameters that control the simulation live in [`micrograph.toml`](./micrograph.toml), loaded by the second cell via `load_config("micrograph.toml", MicrographConfig)` — **edit that file, not this notebook**, to change what gets generated.

## Editing `micrograph.toml`

The TOML file is grouped into tables for readability; `load_config` flattens them, so the table names (`[potential]`, `[microscope]`, etc.) are just organization and don't need to match `MicrographConfig` field names directly:

| Table | Controls |
|---|---|
| `[potential]` | PDB code / assembly, particle box size (`num_pixels`), pixel size, micrograph size |
| `[microscope]` | Accelerating voltage, dose, spherical aberration (`cs`), amplitude contrast (`alpha`) |
| `[defocus]` | Min/max defocus range micrographs are drawn from |
| `[dataset]` | Number of micrographs to generate (`n_micrographs`) |
| `[models]` | Which physics models to use (scattering, aberration, noise, ice, detector) |
| `[ice]` | Ice thickness and how many unique ice cubes to build |
| `[compute]` | Device (`cuda:0`, `cpu`, ...) |

A commented-out line means that field is left at its `MicrographConfig` default — see [`configs/micrograph/micrograph.toml`](../../configs/micrograph/micrograph.toml) for the full set of available fields and their defaults, and [`src/specter/config.py`](../../src/specter/config.py) for authoritative types/units. Common edits:

- **Change the structure**: set `pdb_code` under `[potential]` to a different PDB ID.
- **Generate more micrographs**: change `n_micrographs` under `[dataset]`.
- **Use a different GPU or CPU**: change `device` under `[compute]` (e.g. `"cuda:0"`, `"cuda:1"`, `"cpu"`).
- **Widen/narrow the defocus range**: adjust `defocus_min`/`defocus_max` under `[defocus]` (Å).

After editing the TOML file, re-run the notebook from the top (or just re-run the `load_config(...)` cell and everything below it) to pick up the new values.


In [ ]:
import logging

import matplotlib.pyplot as plt
import torch

import specter
from specter.arrays import radial_profile_2d
from specter.config import load_config, MicrographConfig
from specter.fft import fft2
from specter.filters import butter
from specter.ice import IceBank
from specter.imagegenerator import MicrographGenerator
from specter.pdb import PDB
from specter.plots import plot3d
from specter.potential import PotentialBuilder
from specter.progress import track

%load_ext autoreload
%autoreload 2

In [ ]:
config = load_config("micrograph.toml", MicrographConfig)

In [ ]:
specter.set_verbosity(logging.INFO)

## Create 3D scattering potential

In [ ]:
# fetch pdb file
pdb = PDB(config.pdb_code, assembly=config.assembly, savefolder=config.pdb_savefolder)

In [ ]:
# Potential-building always runs on CPU, independent of config.device (which
# is used later for the generation model) — matches generate_micrograph.py.
pb = PotentialBuilder(config.num_pixels, config.pixel_size, pdb.atomic_numbers)

In [ ]:
with torch.no_grad():
    V = pb(pdb.coordinates).clone()

In [ ]:
plot3d(V)

## Micrograph class

In [ ]:
# Derived values not directly in MicrographConfig
Cs = config.cs * 1e7  # mm -> Å
num_frames = (
    config.num_frames if config.num_frames is not None else int(config.dose_min)
)

In [ ]:
# random defocus
defocus_A = (
    torch.rand(config.n_micrographs) * (config.defocus_max - config.defocus_min)
    + config.defocus_min
)

# Can include 'cs', 'dfu', 'dfv', 'dfang', 'tiltx', 'tilty', 'phaseshift', 'trefoil1, 'trefoil2'
ctf_params = {
    "cs": torch.tensor([Cs] * config.n_micrographs),
    "dfu": defocus_A,
}

In [ ]:
device = config.device  # choose cpu if you run out of cuda memory.

In [ ]:
# Build the bank once — move to GPU before building
icecube_size = (
    config.icecube_size
    if config.icecube_size is not None
    else min(config.num_pixels, int(256 / config.pixel_size))
)
bank = IceBank(
    dx=config.pixel_size,
    n=icecube_size,
    method=config.ice_model,
    num_unique=config.num_unique_icecubes,
    build_batch_size=config.ice_build_batch_size,
).to(device)
bank.build()

In [ ]:
# Coincidence radius in pixels.
# Higher values suppress lower spatial frequencies.
# Set to 0 to recover default Poisson behaviour.
coincidence_radius = config.coincidence_radius_min

crowd_min_distance = (
    config.crowd_min_distance
    if config.crowd_min_distance is not None
    else pdb.max_diameter
)

detector_model = None if config.detector_model == "none" else config.detector_model

model = MicrographGenerator(
    V,
    config.micrograph_size,
    config.pixel_size,
    ctf_params,
    config.energy,
    config.dose_min,
    icemaker=bank,
    ice_thickness=config.ice_thickness,
    scattering_model=config.scattering_model,
    aberration_model=config.aberration_model,
    noise_model=config.noise_model,
    klim=None,
    alpha=config.alpha,
    crowd_min_distance=crowd_min_distance,
    crowd_max_distance_z=config.crowd_max_distance_z,
    water_air_interface=config.water_air_interface,
    pad_fft=config.pad_fft,
    chunk_size=config.chunk_size,
    move_to_cpu=True,
    coincidence_radius=coincidence_radius,
    num_frames=num_frames,
).to(device)

In [ ]:
idx = torch.arange(config.n_micrographs)
images = []
with torch.no_grad():
    for i in track(range(config.n_micrographs), description="Generating micrographs"):
        if i > 0:
            model.regenerate_specimen()
        image = model(torch.tensor([i]))
        images.append(image.detach().cpu())

images = torch.concat(images, dim=0)
images.shape

In [ ]:
fig, axes = plt.subplots(1, 3, dpi=200, constrained_layout=True)
[ax.set(xticks=[], yticks=[]) for ax in axes.ravel()]
ax = axes[0]
ax.imshow(images[0], cmap="grey")
ax.set_title("Image")

ax = axes[1]
ax.imshow(torch.abs(butter(images[0])), cmap="grey")
ax.set_title("Low-pass")

ax = axes[2]
ax.imshow(
    torch.abs(fft2(images[0] - images[0].mean(), shift=True)), cmap="grey", vmax=20000
)
ax.set_title("FFT")
plt.show()

In [ ]:
freqs = torch.fft.fftfreq(config.micrograph_size, config.pixel_size)
freqs = freqs[: config.micrograph_size // 2]
rp = radial_profile_2d(torch.abs(fft2(images[0] - images[0].mean(), shift=True)))
plt.plot(freqs, rp[: config.micrograph_size // 2])
tick_freqs = torch.linspace(freqs[0], freqs[-1], 6)  # Start at index 1 to avoid 1/0
tick_labels = [f"{1 / f:.2f}" for f in tick_freqs]
tick_labels[0] = r"$\infty$"
plt.xticks(tick_freqs, tick_labels)
plt.xlabel("Resolution (A)")
plt.title("FFT radial profile")
plt.show()

In [ ]:
plot3d(model.vol[0].detach().cpu(), cmap="bone")

## Saving output

Uncomment the cell below to save the generated micrograph(s) as an `.mrcs` stack plus a matching `.star` file (same format as `demo-scripts/generate_micrograph.py`).

In [ ]:
# import os
#
# import mrcfile
#
# from specter.cryosparc import create_micrograph_starfile
#
# output_dir = "./output/"
# filename = "micrographs"
# os.makedirs(output_dir, exist_ok=True)
# with mrcfile.new(os.path.join(output_dir, filename + ".mrcs"), overwrite=True) as mrc:
#     mrc.set_data(images.numpy().astype("float32"))
# create_micrograph_starfile(
#     config.n_micrographs,
#     energy=config.energy,
#     pixel_size=config.pixel_size,
#     alpha=config.alpha,
#     ctf_params=ctf_params,
#     folderpath=output_dir,
#     filename=filename,
#     dose_per_angstrom=torch.full((config.n_micrographs,), config.dose_min),
#     coincidence_radius=torch.full((config.n_micrographs,), coincidence_radius),
# )